In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

ModuleNotFoundError: No module named 'cv2'

In [ ]:
dataset/
    images/
        download.jpg
    masks/
        mask1.png
        mask2.png

In [ ]:
import os
import cv2
import numpy as np

image_dir = "dataset/images/"
mask_dir = "dataset/masks/"

IMG_SIZE = 128

images = []
masks = []

for file in os.listdir(image_dir):
    
    img = cv2.imread(image_dir + file, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    
    mask = cv2.imread(mask_dir + file, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = mask / 255.0
    
    images.append(img)
    masks.append(mask)

images = np.array(images)
masks = np.array(masks)

images = np.expand_dims(images, axis=-1)
masks = np.expand_dims(masks, axis=-1)

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

In [ ]:
image_dir = "dataset/images/"
mask_dir = "dataset/masks/"

IMG_SIZE = 128

images = []
masks = []

for filename in os.listdir(image_dir):
    
    img = cv2.imread(image_dir + filename, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    
    mask = cv2.imread(mask_dir + filename, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = mask / 255.0
    
    images.append(img)
    masks.append(mask)

images = np.array(images)
masks = np.array(masks)

images = np.expand_dims(images, axis=-1)
masks = np.expand_dims(masks, axis=-1)

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    images, masks, test_size=0.2, random_state=42
)

In [ ]:
def build_unet(input_shape=(128,128,1)):
    
    inputs = layers.Input(input_shape)
    
    # Encoder
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D()(c1)
    
    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D()(c2)
    
    # Bottleneck
    c3 = layers.Conv2D(256, 3, activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, 3, activation='relu', padding='same')(c3)
    
    # Decoder
    u1 = layers.UpSampling2D()(c3)
    u1 = layers.concatenate([u1, c2])
    c4 = layers.Conv2D(128, 3, activation='relu', padding='same')(u1)
    
    u2 = layers.UpSampling2D()(c4)
    u2 = layers.concatenate([u2, c1])
    c5 = layers.Conv2D(64, 3, activation='relu', padding='same')(u2)
    
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c5)
    
    model = models.Model(inputs, outputs)
    
    return model

model = build_unet()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=8
)

In [ ]:
predictions = model.predict(X_test)

In [ ]:
pred_mask = (predictions > 0.5).astype(np.uint8)

In [ ]:
index = 0

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("Original Image")
plt.imshow(X_test[index].squeeze(), cmap='gray')

plt.subplot(1,3,2)
plt.title("Actual Mask")
plt.imshow(y_test[index].squeeze(), cmap='gray')

plt.subplot(1,3,3)
plt.title("Predicted Mask")
plt.imshow(pred_mask[index].squeeze(), cmap='gray')

plt.show()

In [ ]:
model.save("unet_model.h5")

In [ ]:
model = tf.keras.models.load_model("unet_model.h5")